<a href="https://colab.research.google.com/github/kannisharath/INFO-5731/blob/main/Karrepu_Sharath_Exercise_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 In-class Exercise 5**

**This exercise aims to provide a comprehensive learning experience in text analysis and machine learning techniques, focusing on both text classification and clustering tasks.**

***Please use the text corpus you collected in your last in-class-exercise for this exercise. Perform the following tasks***.

**Expectations**:
*   Students are expected to complete the exercise during lecture period to meet the active participation criteria of the course.
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

**Total points**: 40

**Deadline**: This in-class exercise is due at the end of the day tomorrow, at 11:59 PM.

**Late submissions will have a penalty of 10% of the marks for each day of late submission, and no requests will be answered. Manage your time accordingly.**


## **Question 1 (20 Points)**

The purpose of the question is to practice different machine learning algorithms for **text classification** as well as the performance evaluation. In addition, you are requried to conduct **10 fold cross validation** (https://scikit-learn.org/stable/modules/cross_validation.html) in the training.



The dataset can be download from canvas. The dataset contains two files train data and test data for sentiment analysis in IMDB review, it has two categories: 1 represents positive and 0 represents negative. You need to split the training data into training and validate data (80% for training and 20% for validation, https://towardsdatascience.com/train-test-split-and-cross-validation-in-python-80b61beca4b6) and perform 10 fold cross validation while training the classifier. The final trained model was final evaluated on the test data.


**Algorithms:**

*   MultinominalNB
*   SVM
*   KNN
*   Decision tree
*   Random Forest
*   XGBoost
*   Word2Vec
*   BERT

**Evaluation measurement:**


*   Accuracy
*   Recall
*   Precison
*   F-1 score


In [ ]:
# Write your code here
#Importing the required libraries
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

#Reading the train dataset

In [ ]:
with open("/content/sample_data/stsa-train.txt") as txtf:
    mylist = [line.rstrip('\n') for line in txtf]

labels = []
text = []

for i, line in enumerate(mylist):
    label = mylist[i][0]
    tex = mylist[i][1:]
    labels.append(label)
    text.append(tex)

train_dataset = pd.DataFrame(list(zip(labels, text)),columns =['Reviews', 'Text'])
train_dataset.head()

,Reviews,Text
0,1,"a stirring , funny and finally transporting r..."
1,0,apparently reassembled from the cutting-room ...
2,0,they presume their audience wo n't sit still ...
3,1,this is a visually stunning rumination on lov...
4,1,jonathan parker 's bartleby should have been ...


#Preprocessing of Training Dataset

In [ ]:
!pip install nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import nltk
#nltk.download()
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer,PorterStemmer
from nltk.corpus import stopwords
import re
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess(sentence):
    sentence=str(sentence)
    sentence = sentence.lower()
    sentence=sentence.replace('{html}',"")
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', sentence)
    rem_url=re.sub(r'http\S+', '',cleantext)
    rem_num = re.sub('[0-9]+', '', rem_url)
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(rem_num)
    filtered_words = [w for w in tokens if len(w) > 2 if not w in stopwords.words('english')]
    stem_words=[stemmer.stem(w) for w in filtered_words]
    lemma_words=[lemmatizer.lemmatize(w) for w in stem_words]
    return " ".join(filtered_words)


train_dataset['cleanText']=train_dataset['Text'].map(lambda s:preprocess(s))
train_dataset.head()











,Reviews,Text,cleanText
0,1,"a stirring , funny and finally transporting r...",stirring funny finally transporting imagining ...
1,0,apparently reassembled from the cutting-room ...,apparently reassembled cutting room floor give...
2,0,they presume their audience wo n't sit still ...,presume audience sit still sociology lesson ho...
3,1,this is a visually stunning rumination on lov...,visually stunning rumination love memory histo...
4,1,jonathan parker 's bartleby should have been ...,jonathan parker bartleby end modern office ano...


#Reading the test dataset

In [ ]:
with open("/content/sample_data/stsa-test.txt") as txtf:
    mylist_test_data = [line.rstrip('\n') for line in txtf]

labels_test = []
text_test = []

for i, line in enumerate(mylist_test_data):
    label_test = mylist_test_data[i][0]
    tex_test = mylist_test_data[i][1:]
    labels_test.append(label_test)
    text_test.append(tex_test)

test_dataset = pd.DataFrame(list(zip(labels_test, text_test)),columns =['Reviews', 'Text'])
test_dataset.head()


,Reviews,Text
0,0,"no movement , no yuks , not much of anything ."
1,0,"a gob of drivel so sickly sweet , even the ea..."
2,0,"gangs of new york is an unapologetic mess , w..."
3,0,we never really feel involved with the story ...
4,1,this is one of polanski 's best films .


#Preprocessing of Testing Dataset

In [ ]:
test_dataset['cleanText']=test_dataset['Text'].map(lambda s:preprocess(s))
test_dataset.head()

,Reviews,Text,cleanText
0,0,"no movement , no yuks , not much of anything .",movement yuks much anything
1,0,"a gob of drivel so sickly sweet , even the ea...",gob drivel sickly sweet even eager consumers m...
2,0,"gangs of new york is an unapologetic mess , w...",gangs new york unapologetic mess whose saving ...
3,0,we never really feel involved with the story ...,never really feel involved story ideas remain ...
4,1,this is one of polanski 's best films .,one polanski best films


#TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(lowercase = False, analyzer='word')
tfIDF_train = tfidf_vectorizer.fit_transform(train_dataset["cleanText"]).toarray()
tfIDF_test = tfidf_vectorizer.transform(test_dataset["cleanText"]).toarray()

In [ ]:
x_test = tfIDF_test
y_test = test_dataset["Reviews"]

#Data partitioning

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_valid, y_train, y_valid = train_test_split(tfIDF_train,train_dataset["Reviews"],test_size = 0.2, random_state = 85)

#Testing with Algorithms

#1. MultinominalNB (Multinominal Naive Bayes)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_classifier = MultinomialNB()
nb_model = nb_classifier.fit(x_train, y_train)
predictions_validation_set = nb_classifier.predict(x_valid)


from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
print ("Accuracy of Multinominal  Naive Bayes model  : ", round(accuracy_score(y_valid, predictions_validation_set)*100),"%")
print ("Percision of Multinominal Naive Bayes model  : ", round(precision_score(y_valid, predictions_validation_set, pos_label='0')*100),"%")
print ("Recall of Multinominal Naive Bayes model  : ", round(recall_score(y_valid, predictions_validation_set, pos_label='0')*100),"%")
print ("F1 Score of Multinominal Naive Bayes model  : ", round(f1_score(y_valid, predictions_validation_set, pos_label='0')*100),"%")

Accuracy of Multinominal  Naive Bayes model  :  78 %
Percision of Multinominal Naive Bayes model  :  83 %
Recall of Multinominal Naive Bayes model  :  67 %
F1 Score of Multinominal Naive Bayes model  :  74 %


In [ ]:
from sklearn.metrics import classification_report

classification_Report_naive_bayes = classification_report(y_valid, predictions_validation_set)
print("Classification Report: ", "\n", "\n",classification_Report_naive_bayes)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.83      0.67      0.74       650
           1       0.75      0.88      0.81       734

    accuracy                           0.78      1384
   macro avg       0.79      0.78      0.78      1384
weighted avg       0.79      0.78      0.78      1384



In [ ]:
from sklearn.model_selection import cross_val_score
naive_accuracies_validation = cross_val_score(estimator = nb_classifier, X = x_train, y = y_train, cv = 10)

print(f"Naive Bayes Model  10-fold cross validation score on training set is :  {round(naive_accuracies_validation.mean()*100)}%")

Naive Bayes Model  10-fold cross validation score on training set is :  78%


In [ ]:
predictions_test_set = nb_classifier.predict(x_test)
print ("Accuracy of the Naive Bayes model on test set is : ", round(accuracy_score(y_test, predictions_test_set)*100),"%")
print ("Percision of the Naive Bayes model on validation set is : ", round(precision_score(y_test, predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the Naive Bayes model on validation set is : ", round(recall_score(y_test, predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the Naive Bayes model on validation set is : ", round(f1_score(y_test, predictions_test_set, pos_label='0')*100),"%")

Accuracy of the Naive Bayes model on test set is :  80 %
Percision of the Naive Bayes model on validation set is :  86 %
Recall of the Naive Bayes model on validation set is :  71 %
F1 Score of the Naive Bayes model on validation set is :  78 %


In [ ]:
classification_Report_naive_bayes_Test_data = classification_report(y_test, predictions_test_set)
print("Classification Report: ", "\n", "\n",classification_Report_naive_bayes_Test_data)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.86      0.71      0.78       912
           1       0.75      0.88      0.81       909

    accuracy                           0.80      1821
   macro avg       0.81      0.80      0.80      1821
weighted avg       0.81      0.80      0.80      1821



In [ ]:
naive_accuracies_test = cross_val_score(estimator = nb_classifier, X = x_test, y = y_test, cv = 10)

print(f"Naive Bayes Model 10-fold cross validation score on testing set is :  {round(naive_accuracies_test.mean()*100)}%")

Naive Bayes Model 10-fold cross validation score on testing set is :  73%


#Support Vector Machine

In [ ]:
from sklearn import svm
classifier_svm = svm.SVC()
model_svm = classifier_svm.fit(x_train, y_train)
svm_predictions_validation_set = classifier_svm.predict(x_valid)

In [ ]:
print ("Accuracy of the SVM model on validation set is : ", round(accuracy_score(y_valid, svm_predictions_validation_set)*100),"%")
print ("Percision of the SVM model on validation set is : ", round(precision_score(y_valid, svm_predictions_validation_set, pos_label='0')*100),"%")
print ("Recall of the SVM model on validation set is : ", round(recall_score(y_valid, svm_predictions_validation_set, pos_label='0')*100),"%")
print ("F1 Score of the SVM model on validation set is : ", round(f1_score(y_valid, svm_predictions_validation_set, pos_label='0')*100),"%")

Accuracy of the SVM model on validation set is :  79 %
Percision of the SVM model on validation set is :  80 %
Recall of the SVM model on validation set is :  75 %
F1 Score of the SVM model on validation set is :  77 %


In [ ]:
from sklearn.metrics import classification_report

svm_validation_Classification_report = classification_report(y_valid, svm_predictions_validation_set)
print("Classification Report: ", "\n", "\n",svm_validation_Classification_report)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.80      0.75      0.77       650
           1       0.79      0.83      0.81       734

    accuracy                           0.79      1384
   macro avg       0.79      0.79      0.79      1384
weighted avg       0.79      0.79      0.79      1384



In [ ]:
from sklearn.model_selection import cross_val_score
svm_accuracies_validation = cross_val_score(estimator = classifier_svm, X = x_train, y = y_train, cv = 10)

print(f"SVM Model  10-fold cross validation score on training set is :  {round(svm_accuracies_validation.mean()*100)}%")

SVM Model  10-fold cross validation score on training set is :  77%


In [ ]:
svm_predictions_test_set = classifier_svm.predict(x_test)
print ("Accuracy of the SVM model on test set is : ", round(accuracy_score(y_test, svm_predictions_test_set)*100),"%")
print ("Percision of the SVM model on validation set is : ", round(precision_score(y_test, svm_predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the SVM model on validation set is : ", round(recall_score(y_test, svm_predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the SVM model on validation set is : ", round(f1_score(y_test, svm_predictions_test_set, pos_label='0')*100),"%")

Accuracy of the SVM model on test set is :  80 %
Percision of the SVM model on validation set is :  82 %
Recall of the SVM model on validation set is :  76 %
F1 Score of the SVM model on validation set is :  79 %


In [ ]:
svm_predictions_test_set = classifier_svm.predict(x_test)
print ("Accuracy of the SVM model on test set is : ", round(accuracy_score(y_test, svm_predictions_test_set)*100),"%")
print ("Percision of the SVM model on validation set is : ", round(precision_score(y_test, svm_predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the SVM model on validation set is : ", round(recall_score(y_test, svm_predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the SVM model on validation set is : ", round(f1_score(y_test, svm_predictions_test_set, pos_label='0')*100),"%")

In [ ]:
svm_test_validation_Classification_report = classification_report(y_test, svm_predictions_test_set)
print("Classification Report: ", "\n", "\n",svm_test_validation_Classification_report)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.82      0.76      0.79       912
           1       0.78      0.84      0.81       909

    accuracy                           0.80      1821
   macro avg       0.80      0.80      0.80      1821
weighted avg       0.80      0.80      0.80      1821



In [ ]:
svm_accuracies_test = cross_val_score(estimator = classifier_svm, X = x_test, y = y_test, cv = 10)

print(f"SVM Model 10-fold cross validation score on testing set is :  {round(svm_accuracies_test.mean()*100)}%")

SVM Model 10-fold cross validation score on testing set is :  72%


#KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

classifier_knn = KNeighborsClassifier(n_neighbors = 15)
model_knn = classifier_knn.fit(x_train, y_train)
knn_predictions_validation_set = classifier_knn.predict(x_valid)

print ("Accuracy of the KNN model on validation set is : ", round(accuracy_score(y_valid, knn_predictions_validation_set)*100),"%")
print ("Percision of the KNN model on validation set is : ", round(precision_score(y_valid, knn_predictions_validation_set, pos_label='0')*100),"%")
print ("Recall of the KNN model on validation set is : ", round(recall_score(y_valid, knn_predictions_validation_set, pos_label='0')*100),"%")
print ("F1 Score of the KNN model on validation set is : ", round(f1_score(y_valid, knn_predictions_validation_set, pos_label='0')*100),"%")

Accuracy of the KNN model on validation set is :  70 %
Percision of the KNN model on validation set is :  64 %
Recall of the KNN model on validation set is :  80 %
F1 Score of the KNN model on validation set is :  71 %


In [ ]:
from sklearn.metrics import classification_report

knn_validation_Classification_report = classification_report(y_valid, knn_predictions_validation_set)
print("Classification Report: ", "\n", "\n",knn_validation_Classification_report)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.64      0.80      0.71       650
           1       0.78      0.60      0.68       734

    accuracy                           0.70      1384
   macro avg       0.71      0.70      0.70      1384
weighted avg       0.71      0.70      0.69      1384



In [ ]:
from sklearn.model_selection import cross_val_score
knn_accuracies_validation = cross_val_score(estimator = classifier_knn, X = x_train, y = y_train, cv = 10)

print(f"KNN Model  10-fold cross validation score on training set is :  {round(knn_accuracies_validation.mean()*100)}%")

KNN Model  10-fold cross validation score on training set is :  71%


In [ ]:
knn_predictions_test_set = classifier_knn.predict(x_test)
print ("Accuracy of the KNN model on test set is : ", round(accuracy_score(y_test, knn_predictions_test_set)*100),"%")
print ("Percision of the KNN model on validation set is : ", round(precision_score(y_test, knn_predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the KNN model on validation set is : ", round(recall_score(y_test, knn_predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the KNN model on validation set is : ", round(f1_score(y_test, knn_predictions_test_set, pos_label='0')*100),"%")

Accuracy of the KNN model on test set is :  73 %
Percision of the KNN model on validation set is :  68 %
Recall of the KNN model on validation set is :  85 %
F1 Score of the KNN model on validation set is :  76 %


In [ ]:
knn_test_validation_Classification_report = classification_report(y_test, knn_predictions_test_set)
print("Classification Report: ", "\n", "\n",knn_test_validation_Classification_report)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.68      0.85      0.76       912
           1       0.80      0.60      0.69       909

    accuracy                           0.73      1821
   macro avg       0.74      0.73      0.72      1821
weighted avg       0.74      0.73      0.72      1821



In [ ]:
knn_accuracies_test = cross_val_score(estimator = classifier_knn, X = x_test, y = y_test, cv = 10)

print(f"KNN Model 10-fold cross validation score on testing set is :  {round(knn_accuracies_test.mean()*100)}%")

KNN Model 10-fold cross validation score on testing set is :  63%


#Decison Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

classifier_dt = DecisionTreeClassifier()
model_dt = classifier_dt.fit(x_train, y_train)
dt_predictions_validation_set = classifier_dt.predict(x_valid)

print ("Accuracy of the Decison Tree Classifier model on validation set is : ", round(accuracy_score(y_valid, dt_predictions_validation_set)*100),"%")
print ("Percision of the Decison Tree Classifier model on validation set is : ", round(precision_score(y_valid, dt_predictions_validation_set, pos_label='0')*100),"%")
print ("Recall of the Decison Tree Classifier model on validation set is : ", round(recall_score(y_valid, dt_predictions_validation_set, pos_label='0')*100),"%")
print ("F1 Score of the Decison Tree Classifier model on validation set is : ", round(f1_score(y_valid, dt_predictions_validation_set, pos_label='0')*100),"%")

Accuracy of the Decison Tree Classifier model on validation set is :  67 %
Percision of the Decison Tree Classifier model on validation set is :  63 %
Recall of the Decison Tree Classifier model on validation set is :  70 %
F1 Score of the Decison Tree Classifier model on validation set is :  66 %


In [ ]:
from sklearn.metrics import classification_report

dt_validation_Classification_report = classification_report(y_valid, dt_predictions_validation_set)
print("Classification Report: ", "\n", "\n",dt_validation_Classification_report)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.63      0.70      0.66       650
           1       0.70      0.64      0.67       734

    accuracy                           0.67      1384
   macro avg       0.67      0.67      0.67      1384
weighted avg       0.67      0.67      0.67      1384



In [ ]:
dt_predictions_test_set = classifier_dt.predict(x_test)
print ("Accuracy of the Decison Tree Classifier model on test set is : ", round(accuracy_score(y_test, dt_predictions_test_set)*100),"%")
print ("Percision of the Decison Tree Classifier model on validation set is : ", round(precision_score(y_test, dt_predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the Decison Tree Classifier model on validation set is : ", round(recall_score(y_test, dt_predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the Decison Tree Classifier model on validation set is : ", round(f1_score(y_test, dt_predictions_test_set, pos_label='0')*100),"%")

Accuracy of the Decison Tree Classifier model on test set is :  67 %
Percision of the Decison Tree Classifier model on validation set is :  67 %
Recall of the Decison Tree Classifier model on validation set is :  69 %
F1 Score of the Decison Tree Classifier model on validation set is :  68 %


In [ ]:
dt_validation_Classification_report_test = classification_report(y_test, dt_predictions_test_set)
print("Classification Report: ", "\n", "\n",dt_validation_Classification_report_test)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.67      0.69      0.68       912
           1       0.68      0.66      0.67       909

    accuracy                           0.67      1821
   macro avg       0.67      0.67      0.67      1821
weighted avg       0.67      0.67      0.67      1821



In [ ]:
dt_accuracies_test = cross_val_score(estimator = classifier_dt, X = x_test, y = y_test, cv = 10)

print(f"Decison Tree Classifier Model 10-fold cross validation score on testing set is :  {round(dt_accuracies_test.mean()*100)}%")

Decison Tree Classifier Model 10-fold cross validation score on testing set is :  63%


In [ ]:
from sklearn.model_selection import cross_val_score
dt_accuracies_validation = cross_val_score(estimator = classifier_dt, X = x_train, y = y_train, cv = 10)

print(f"Decison Tree Classifier Model  10-fold cross validation score on training set is :  {round(dt_accuracies_validation.mean()*100)}%")

Decison Tree Classifier Model  10-fold cross validation score on training set is :  65%


#Randomforest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier_rf = RandomForestClassifier()
model_rf = classifier_rf.fit(x_train, y_train)
rf_predictions_validation_set = classifier_rf.predict(x_valid)

print ("Accuracy of the Random Forest Classifier model on validation set is : ", round(accuracy_score(y_valid, rf_predictions_validation_set)*100),"%")
print ("Percision of the Random Forest Classifier model on validation set is : ", round(precision_score(y_valid, rf_predictions_validation_set, pos_label='0')*100),"%")
print ("Recall of the Random Forest Classifier model on validation set is : ", round(recall_score(y_valid, rf_predictions_validation_set, pos_label='0')*100),"%")
print ("F1 Score of the Random Forest Classifier model on validation set is : ", round(f1_score(y_valid, rf_predictions_validation_set, pos_label='0')*100),"%")

Accuracy of the Random Forest Classifier model on validation set is :  74 %
Percision of the Random Forest Classifier model on validation set is :  71 %
Recall of the Random Forest Classifier model on validation set is :  76 %
F1 Score of the Random Forest Classifier model on validation set is :  73 %


In [ ]:
from sklearn.metrics import classification_report

rf_validation_Classification_report = classification_report(y_valid, rf_predictions_validation_set)
print("Classification Report: ", "\n", "\n",rf_validation_Classification_report)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.71      0.76      0.73       650
           1       0.77      0.73      0.75       734

    accuracy                           0.74      1384
   macro avg       0.74      0.74      0.74      1384
weighted avg       0.74      0.74      0.74      1384



In [ ]:
from sklearn.model_selection import cross_val_score
rf_accuracies_validation = cross_val_score(estimator = classifier_rf, X = x_train, y = y_train, cv = 10)

print(f"Decison Random Forest Model  10-fold cross validation score on training set is :  {round(rf_accuracies_validation.mean()*100)}%")

Decison Random Forest Model  10-fold cross validation score on training set is :  72%


In [ ]:
rf_predictions_test_set = classifier_rf.predict(x_test)
print ("Accuracy of the Random Forest Classifier model on test set is : ", round(accuracy_score(y_test, rf_predictions_test_set)*100),"%")
print ("Percision of the Random Forest Classifier model on validation set is : ", round(precision_score(y_test, rf_predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the Random Forest Classifier model on validation set is : ", round(recall_score(y_test, rf_predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the Random Forest Classifier model on validation set is : ", round(f1_score(y_test, rf_predictions_test_set, pos_label='0')*100),"%")

Accuracy of the Random Forest Classifier model on test set is :  76 %
Percision of the Random Forest Classifier model on validation set is :  75 %
Recall of the Random Forest Classifier model on validation set is :  79 %
F1 Score of the Random Forest Classifier model on validation set is :  77 %


In [ ]:
rf_validation_Classification_report_test = classification_report(y_test, rf_predictions_test_set)
print("Classification Report: ", "\n", "\n",rf_validation_Classification_report_test)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.75      0.79      0.77       912
           1       0.78      0.73      0.75       909

    accuracy                           0.76      1821
   macro avg       0.76      0.76      0.76      1821
weighted avg       0.76      0.76      0.76      1821



In [ ]:
rf_accuracies_test = cross_val_score(estimator = classifier_rf, X = x_test, y = y_test, cv = 10)

print(f"Random Forest Classifier Model 10-fold cross validation score on testing set is :  {round(rf_accuracies_test.mean()*100)}%")

Random Forest Classifier Model 10-fold cross validation score on testing set is :  66%


In [ ]:
from sklearn.model_selection import cross_val_score
rf_accuracies_validation = cross_val_score(estimator = classifier_rf, X = x_train, y = y_train, cv = 10)

print(f"Decison Random Forest Model  10-fold cross validation score on training set is :  {round(rf_accuracies_validation.mean()*100)}%")

rf_predictions_test_set = classifier_rf.predict(x_test)
print ("Accuracy of the Random Forest Classifier model on test set is : ", round(accuracy_score(y_test, rf_predictions_test_set)*100),"%")
print ("Percision of the Random Forest Classifier model on validation set is : ", round(precision_score(y_test, rf_predictions_test_set, pos_label='0')*100),"%")
print ("Recall of the Random Forest Classifier model on validation set is : ", round(recall_score(y_test, rf_predictions_test_set, pos_label='0')*100),"%")
print ("F1 Score of the Random Forest Classifier model on validation set is : ", round(f1_score(y_test, rf_predictions_test_set, pos_label='0')*100),"%")

cr_rf_test = classification_report(y_test, rf_predictions_test_set)
print("Classification Report: ", "\n", "\n",cr_rf_test)

rf_accuracies_test = cross_val_score(estimator = classifier_rf, X = x_test, y = y_test, cv = 10)

print(f"Random Forest Classifier Model 10-fold cross validation score on testing set is :  {round(rf_accuracies_test.mean()*100)}%")

Decison Random Forest Model  10-fold cross validation score on training set is :  72%
Accuracy of the Random Forest Classifier model on test set is :  76 %
Percision of the Random Forest Classifier model on validation set is :  75 %
Recall of the Random Forest Classifier model on validation set is :  79 %
F1 Score of the Random Forest Classifier model on validation set is :  77 %
Classification Report:  
 
               precision    recall  f1-score   support

           0       0.75      0.79      0.77       912
           1       0.78      0.73      0.75       909

    accuracy                           0.76      1821
   macro avg       0.76      0.76      0.76      1821
weighted avg       0.76      0.76      0.76      1821

Random Forest Classifier Model 10-fold cross validation score on testing set is :  66%


#XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

classifier_xgb = XGBClassifier()
model_xgb = classifier_xgb.fit(x_train, y_train)
xgb_predictions_validation_set = classifier_xgb.predict(x_valid)

print("Accuracy of the XGBoost Classifier model on validation set is : ", round(accuracy_score(y_valid, xgb_predictions_validation_set)*100),"%")
print("Precision of the XGBoost Classifier model on validation set is : ", round(precision_score(y_valid, xgb_predictions_validation_set, pos_label=0)*100),"%")
print("Recall of the XGBoost Classifier model on validation set is : ", round(recall_score(y_valid, xgb_predictions_validation_set, pos_label=0)*100),"%")
print("F1 Score of the XGBoost Classifier model on validation set is : ", round(f1_score(y_valid, xgb_predictions_validation_set, pos_label=0)*100),"%")


Accuracy of the XGBoost Classifier model on validation set is :  69 %
Precision of the XGBoost Classifier model on validation set is :  71 %
Recall of the XGBoost Classifier model on validation set is :  56 %
F1 Score of the XGBoost Classifier model on validation set is :  63 %


In [ ]:
from sklearn.metrics import classification_report

cr_xgb_validation = classification_report(y_valid, xgb_predictions_validation_set)
print("Classification Report: ", "\n", "\n",cr_xgb_validation)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.71      0.56      0.63       650
           1       0.67      0.80      0.73       734

    accuracy                           0.69      1384
   macro avg       0.69      0.68      0.68      1384
weighted avg       0.69      0.69      0.68      1384



In [ ]:
from sklearn.model_selection import cross_val_score
xgb_accuracies_validation = cross_val_score(estimator = classifier_xgb, X = x_train, y = y_train, cv = 10)

print(f"XGBoost Model  10-fold cross validation score on training set is :  {round(xgb_accuracies_validation.mean()*100)}%")

XGBoost Model  10-fold cross validation score on training set is :  68%


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Assuming y_test is currently an array of strings, we convert it to integers
y_test = np.array(y_test).astype(int)  # Make sure y_test is of type int, similar to xgb_predictions_test_set

# Making predictions
xgb_predictions_test_set = classifier_xgb.predict(x_test)

# Now, compute the metrics
print("Accuracy of the XGBoost Classifier model on test set is: ", round(accuracy_score(y_test, xgb_predictions_test_set) * 100), "%")
print("Precision of the XGBoost Classifier model on test set is: ", round(precision_score(y_test, xgb_predictions_test_set, pos_label=0) * 100), "%")
print("Recall of the XGBoost Classifier model on test set is: ", round(recall_score(y_test, xgb_predictions_test_set, pos_label=0) * 100), "%")
print("F1 Score of the XGBoost Classifier model on test set is: ", round(f1_score(y_test, xgb_predictions_test_set, pos_label=0) * 100), "%")


Accuracy of the XGBoost Classifier model on test set is:  70 %
Precision of the XGBoost Classifier model on test set is:  74 %
Recall of the XGBoost Classifier model on test set is:  62 %
F1 Score of the XGBoost Classifier model on test set is:  67 %


In [ ]:
cr_xgb_test = classification_report(y_test, xgb_predictions_test_set)
print("Classification Report: ", "\n", "\n",cr_xgb_test)

Classification Report:  
 
               precision    recall  f1-score   support

           0       0.74      0.62      0.67       912
           1       0.67      0.79      0.73       909

    accuracy                           0.70      1821
   macro avg       0.71      0.70      0.70      1821
weighted avg       0.71      0.70      0.70      1821



In [ ]:
xgb_accuracies_test = cross_val_score(estimator = classifier_xgb, X = x_test, y = y_test, cv = 10)

print(f"XGBoost Classifier Model 10-fold cross validation score on testing set is :  {round(xgb_accuracies_test.mean()*100)}%")

XGBoost Classifier Model 10-fold cross validation score on testing set is :  64%


## **Question 2 (20 Points)**

The purpose of the question is to practice different machine learning algorithms for **text clustering**.

Please downlad the dataset by using the following link.  https://www.kaggle.com/PromptCloudHQ/amazon-reviews-unlocked-mobile-phones
(You can also use different text data which you want)

**Apply the listed clustering methods to the dataset:**
*   K-means
*   DBSCAN
*   Hierarchical clustering
*   Word2Vec
*   BERT

You can refer to of the codes from  the follwing link below.
https://www.kaggle.com/karthik3890/text-clustering

In [ ]:
!pip install your_module

ERROR: Could not find a version that satisfies the requirement your_module (from versions: none)
ERROR: No matching distribution found for your_module


In [ ]:
# Write your code here
df = pd.read_csv('Amazon_Unlocked_Mobile.csv')

df['Reviews']=df['Reviews'].map(lambda s:preprocess(s))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Amazon_Unlocked_Mobile.csv'

In [ ]:
# Write your code here
import pandas as pd
df = pd.read_csv('/content/sample_data/Amazon_Unlocked_Mobile.csv')

df['Reviews']=df['Reviews'].map(lambda s:preprocess(s))
df.head()

NameError: name 'preprocess' is not defined

In [ ]:
# TF-IDF VECTORIZATION

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer()
tfidf_vects = tfidf_vect.fit_transform(df['Reviews'].values.astype('U'))
names= tfidf_vect.get_feature_names()

NameError: name 'df' is not defined

In [ ]:
## ELBOW METHOD

from sklearn.cluster import KMeans
wcss = []
for i in range(2,12):
    kmeans = KMeans(n_clusters = i, init = "k-means++", random_state = 101)
    kmeans.fit(tfidf_vects)
    wcss.append(kmeans.inertia_)

plt.figure(figsize = (11,6))
plt.plot(range(2,12), wcss, marker = "o")
plt.title ("The Elbow Method")
plt.xlabel("Number of clusters")
plt.ylabel("WCSS")

In [ ]:
#forming 6 clusters
from sklearn.cluster import KMeans
model = KMeans(n_clusters = 6,init='k-means++',max_iter=10000, random_state=50)
model.fit(tfidf_vects)
from collections import Counter
Counter(model.labels_)

In [ ]:
# Clusters containing words with maximum strength
top_words = 7
centroids = model.cluster_centers_.argsort()[:, ::-1]
for cluster_num in range(6):
    key_features = [names[i] for i in centroids[cluster_num, :top_words]]
    print('Cluster '+str(cluster_num+1))
    print('Top Words:', key_features)

In [ ]:
cluster_center=model.cluster_centers_
cluster_center

#DBSCAN

In [ ]:
reviews=[]
for i in df['Reviews']:
    reviews.append(str(i).split())
import gensim
w2v_model=gensim.models.Word2Vec(reviews, size=100, workers=4)

import numpy as np
vectors = []
for i in reviews:
    vector = np.zeros(100)
    count = 0
    for word in i:
        try:
            vec = w2v_model.wv[word]
            vector += vec
            count += 1
        except:
            pass
    vector /= count
    vectors.append(vector)
vectors = np.array(vectors)
vectors = np.nan_to_num(vectors)

In [ ]:
from sklearn.cluster import DBSCAN
minPts = 2 * 100
# Lower bound function
def lower_bound(nums, target):
    l, r = 0, len(nums) - 1
    # Binary searching
    while l <= r:
        mid = int(l + (r - l) / 2)
        if nums[mid] >= target:
            r = mid - 1
        else:
            l = mid + 1
    return l

def compute200thnearestneighbour(x, data):
    dists = []
    for val in data:
      # computing distances
        dist = np.sum((x - val) **2 )
        if(len(dists) == 200 and dists[199] > dist):
            l = int(lower_bound(dists, dist))
            if l < 200 and l >= 0 and dists[l] > dist:
                dists[l] = dist
        else:
            dists.append(dist)
            dists.sort()

# Dist 199 contains the distance of 200th nearest neighbour.
    return dists[199]

vectors.shape

In [ ]:
# Computing the 200th nearest neighbour distance of some point the dataset:
twohundrethneigh = []
for val in vectors[:1000]:
    twohundrethneigh.append( compute200thnearestneighbour(val, vectors[:1000]) )
twohundrethneigh.sort()

In [ ]:
# Plotting for the Elbow Method :
%matplotlib inline
from matplotlib import pyplot as plt
plt.figure(figsize=(14,4))
plt.title("Elbow Method for Finding the right Eps hyperparameter")
plt.plot([x for x in range(len(twohundrethneigh))], twohundrethneigh)
plt.xlabel("Number of points")
plt.ylabel("Distance of 200th Nearest Neighbour")
plt.show()

In [ ]:
# Create the model
model_dbs = DBSCAN(eps = 5, min_samples = minPts)
model_dbs.fit(vectors)

In [ ]:
df_dbs = df
df_dbs["DBS Cluster Label"] = model_dbs.labels_
df_dbs

#Hierarchical clustering

In [ ]:
import scipy
from scipy.cluster import hierarchy
dendro=hierarchy.dendrogram(hierarchy.linkage(vectors,method='ward'))
plt.axhline(y=20)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

cluster = AgglomerativeClustering(n_clusters=3, affinity='euclidean', linkage='ward')  #took n=3 from dendrogram curve
Agg=cluster.fit_predict(vectors)


In [ ]:
df['AVG-W2V Clus Label'] = cluster.labels_
df.head()

In [ ]:
hier_df = df # Give the labels and group to count the number of data in each clusters.
hier_df["Hierarchial Cluster Labels"] = cluster.labels_
hier_df.groupby(["Hierarchial Cluster Labels"])["Reviews"].count()

**In one paragraph, please compare the results of K-means, DBSCAN, Hierarchical clustering, Word2Vec, and BERT.**

**Write your response here:**

.

.

.

.

.




# Mandatory Question

**Important: Reflective Feedback on this exercise**

Please provide your thoughts and feedback on the exercises you completed in this assignment.


**(Your submission will not be graded if this question is left unanswered)**



In [ ]:
# Your answer here (no code for this question, write down your answer as detail as possible for the above questions):

'''
The question 2 is getting error due to out of memmory. i have faced the issues.





'''